In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
haejeg1_ice_org_j_path = kagglehub.dataset_download('haejeg1/ice-org-j')

100%|██████████| 576M/576M [00:40<00:00, 14.8MB/s]

Extracting files...


In [ ]:
path = kagglehub.dataset_download('justin2028/us-immigration-statistics-1980-2021')

100%|██████████| 1.42k/1.42k [00:00<00:00, 2.69MB/s]

Extracting files...


In [ ]:
import pandas as pd

In [ ]:
# Create dataframes from ICE data
import os

files = os.listdir(haejeg1_ice_org_j_path)
print(f"Files in dataset: {files}")

arrests_df = pd.read_csv(os.path.join(haejeg1_ice_org_j_path, files[1]))
removals_df = pd.read_csv(os.path.join(haejeg1_ice_org_j_path, files[2]))
classifications_df = pd.read_csv(os.path.join(haejeg1_ice_org_j_path, files[3]))
detentions_df = pd.read_csv(os.path.join(haejeg1_ice_org_j_path, files[0]))

Files in dataset: ['detentions_10yr.csv', 'arrests_10yr.csv', 'removals_10yr.csv', 'classifications_10yr.csv']


/tmp/ipykernel_912/2378758448.py:8: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  removals_df = pd.read_csv(os.path.join(haejeg1_ice_org_j_path, files[2]))
/tmp/ipykernel_912/2378758448.py:9: DtypeWarning: Columns (9,17,29,30,32,38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  classifications_df = pd.read_csv(os.path.join(haejeg1_ice_org_j_path, files[3]))


In [ ]:
# Create dataframes from immigration data
import os

files = os.listdir(path)
print(f"Files in dataset: {files}")

immigration_df = pd.read_csv(os.path.join(path, files[0]))

Files in dataset: ['US Immigration Statistics (Ver 1.14.26).csv']


In [ ]:
arrests_df.columns

# Get the most recent and the oldest apprehension dates
print(arrests_df['Apprehension Date'].max())
print(arrests_df['Apprehension Date'].min())

# According to this, we have a 10 year span from 2011 October all the way until the end of 2021, we'll have to now fix the immigration data to fit these dates

2021-12-31
2011-10-01


In [ ]:
# Adjust immigration data to fit dates of ICE data.

immigration_df = immigration_df[immigration_df['Year'] <= 2021]

# Since immigration data is by year, October - December doesn't seem to be enough data unless we make assumptions for the rest of the year
# The presidency didn't change in 2012 as Barack Obama was re-elected.
# Therefore, we should remove 2011 from both datasets.

immigration_df = immigration_df[immigration_df['Year'] >= 2012]
immigration_df

,Year,Immigrants Obtaining Lawful Permanent Resident Status,Refugee Arrivals,Noncitizen Apprehensions,Noncitizen Removals,Noncitizen Returns
32,2012.0,"1,031,631","58,179","795,735","415,579","231,105"
33,2013.0,"990,553","69,909","786,223","432,201","178,973"
34,2014.0,"1,016,518","69,975","805,334","405,026","163,836"
35,2015.0,"1,051,031","69,920","596,560","324,303","129,636"
36,2016.0,"1,183,505","84,989","683,782","332,263","106,479"
37,2017.0,"1,127,167","53,691","607,677","284,298","100,454"
38,2018.0,"1,096,611","22,405","739,486","327,554","159,960"
39,2019.0,"1,031,765","29,916","1,175,841","347,183","171,125"
40,2020.0,"707,362","11,840","609,265","237,861","167,453"
41,2021.0,"740,002","11,454","1,865,379","89,191","178,227"


### Observations

Although Barack Obama was the president throughout 2012 - 2016, there seems to be an outlier at the  year 2013, with only 990k immigrants obtaining lawful permanent residency status. With this data, we can also see that there was a sudden decline in 2020 - 2021, where COVID could have played a part.

In [ ]:
# Now reorganizing datasets from ICE data to match the dates
arrests_df = arrests_df[arrests_df['Apprehension Date'] >= '2012-01-01']

removals_df = removals_df[removals_df['Apprehension Date'] >= '2012-01-01']

print(removals_df['Apprehension Date'].max())
print(removals_df['Apprehension Date'].min())

# More data inconsistencies, I guess apprehension dates for removals_df weren't cut off properly?

removals_df = removals_df[removals_df['Apprehension Date'] <= '2021-12-31']

print(removals_df['Apprehension Date'].max())
print(removals_df['Apprehension Date'].min())

# No december 31st?

2023-10-21 12:38:08
2012-01-01 00:00:00
2021-12-30 16:00:00
2012-01-01 00:00:00


After analysis and further research on the data of 'Yearly refugee arrivals and other data from the Department of Homeland Security', could be an inaccurate data to compare arrests/removals activity to- as lawful permanent residency can take years to obtain.

After further research, I was able to find a dataset named 'Global Population and Migration dataset', which uses data from the World Bank API- from the United Nations Population Division. A problem with this new dataset is that it's estimated to an extent, and also theoretically tries to include illegal immigrants too.

In [ ]:
haejeg1_migration = kagglehub.dataset_download('hashimkhanwazir/global-population-and-migration-dataset')

100%|██████████| 127k/127k [00:00<00:00, 278kB/s]

Extracting files...


In [ ]:
# Create dataframes from Migration data
import os

files = os.listdir(haejeg1_migration)
print(f"Files in dataset: {files}")

migration_df = pd.read_csv(os.path.join(haejeg1_migration, files[0]))

Files in dataset: ['world_pop_mig_186_countries.csv']


In [ ]:
# Sort with only US migration
migration_df = migration_df[migration_df['country'] == 'United States']
migration_df = migration_df[migration_df['year'] >= 2012]
migration_df = migration_df[migration_df['year'] <= 2021]
migration_df

,country,year,population,netMigration,population_in_millions
11458,United States,2021,332048977.0,561580.0,332
11459,United States,2020,331526933.0,675560.0,331
11460,United States,2019,328329953.0,1158444.0,328
11461,United States,2018,326838199.0,1200796.0,326
11462,United States,2017,325122128.0,1377630.0,325
11463,United States,2016,323071755.0,1449371.0,323
11464,United States,2015,320738994.0,1221849.0,320
11465,United States,2014,318386329.0,1250914.0,318
11466,United States,2013,316059947.0,1320840.0,316
11467,United States,2012,313877662.0,1323368.0,313


In [ ]:
# Export all the new datasets
arrests_df.to_csv('arrests.csv')
removals_df.to_csv('removals.csv')
migration_df.to_csv('net_migrations.csv')
immigration_df.to_csv('immigrations.csv')